# Kairos: existing 14B adapter, inference-only repair comparison

No training, optimizer updates, paid API, paid upgrade or Drive mount. Compare the saved full100 adapter with the same frozen quantized base. These twelve cases are historically exposed and independently qualified, excluded from this adapter's training/development. Three have no captured initial context; retain them as unavailable. The original four qualification exclusions remain recorded. No new holdout or leaderboard claim.

The notebook captures full generations. Independent patch checks run separately on the Mac in disposable Docker environments. No generated patch is executed here.

Run cells sequentially. Transfer verification precedes the roughly 9.93 GB public model download. Preserve errors and partial outputs. Never rerun an uncertain dispatch.


In [ ]:
import os, sys, json, hashlib, platform, shutil, base64
from pathlib import Path
from IPython.display import display, HTML

if platform.system() != 'Linux':
    raise RuntimeError('Use the approved free Colab GPU runtime, not the Mac.')
WORK = Path('/content/kairos-eval020')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
PROBE_BYTES = b'{"payload":"Public evidence transfer check.","training":false,"version":"kairos-evaluation020-probe-v1"}\n'
PROBE = WORK / 'probe-origin' / 'kairos-eval020-probe.json'
PROBE.parent.mkdir(exist_ok=True)
if PROBE.exists() and PROBE.read_bytes() != PROBE_BYTES:
    raise RuntimeError('Preserve unexpected existing probe; inspect it.')
PROBE.write_bytes(PROBE_BYTES)
print({'probe_bytes': len(PROBE_BYTES), 'probe_sha256': hashlib.sha256(PROBE_BYTES).hexdigest(), 'training_runs': 0})
print('KAIROS_PROBE_JSON ' + PROBE_BYTES.decode().strip())
link = 'data:application/json;base64,' + base64.b64encode(PROBE_BYTES).decode()
display(HTML('<a download="kairos-eval020-probe.json" href="' + link + '">Download transfer probe</a>'))


## Upload only the approved existing files

Upload `adapter_model.safetensors`, `adapter_config.json`, frozen `prompts.json`, and the downloaded transfer probe. Never upload private-bindings.json, credentials, browser state, a dataset with reference repairs or base weights. If a native download fails, the visible probe JSON can be saved locally with its exact trailing newline, hashed, then uploaded. This is the same public transfer probe.


In [ ]:
from google.colab import files
EXPECTED = {'prompts.json': {'bytes': 59716, 'sha256': '0cc835a7ad5d1e1a1e31dcad5e35cc2d65e81497936bd305ece9a7451ce5d511'}, 'adapter_config.json': {'bytes': 1122, 'sha256': 'fd601cfbe47e51592c847d96874632be21c5934a4c6584ff378aa84fa5fadf76'}, 'adapter_model.safetensors': {'bytes': 25191384, 'sha256': 'e077393d7464db525ef90216590813620324e60355493f28faf8f777dba57404'}, 'kairos-eval020-probe.json': {'bytes': 105, 'sha256': '08d79dbbc80276fe091438332a6ba28a96f41a112db8409d72cfd32efd7c725f'}}
existing = [name for name in EXPECTED if (WORK / name).exists() and name != PROBE.name]
# The origin probe is kept in a separate directory; the uploaded copy must round-trip unchanged.
if not existing:
    uploaded = files.upload()
    if set(uploaded) != set(EXPECTED):
        raise RuntimeError('Upload membership differs; stop before weights.')
for name, item in EXPECTED.items():
    path = WORK / name
    if not path.is_file() or path.stat().st_size != item['bytes'] or hashlib.sha256(path.read_bytes()).hexdigest() != item['sha256']:
        raise RuntimeError('Input byte hash differs: ' + name)
print({'transfer_verified': True, 'input_files': sorted(EXPECTED), 'training_runs': 0})


## Free GPU and reviewed inference dependencies

This downloads only public packages from PyPI. It preserves the installed CUDA PyTorch and records its version; both arms use the same inference environment. The original training environment remains separately recorded. Stop if CUDA or free storage is unavailable; do not purchase an upgrade or substitute a model.


In [ ]:
import subprocess
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 14 * 1024**3:
    raise RuntimeError('A free CUDA GPU with at least 14 GiB is required. No paid upgrade.')
if shutil.disk_usage('/content').free < 16 * 1024**3:
    raise RuntimeError('Insufficient cloud scratch reserve.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    'transformers==4.57.6', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2'], check=True)
print({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'total_vram_bytes': torch.cuda.get_device_properties(0).total_memory})


## Capture each inference once

At most eighteen model dispatches: nine available cases, base and trained adapter. Three unavailable cases remain in the twelve-case denominator. Greedy decoding, 4,096 total tokens, 1,024 maximum output tokens and a 180-second generation stopping criterion are identical. Context is never truncated. The wall-clock supervisor stops after eighty minutes, leaving any interrupted dispatch unknown. Do not rerun the model cell after interruption.


In [ ]:
RUNNER_SOURCE = '"""Inference-only paired comparison. Never trains, grades repairs or retries unknown dispatches."""\nfrom __future__ import annotations\n\nimport argparse\nfrom contextlib import nullcontext\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport tempfile\nimport time\n\nBASE = \'unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit\'\nREVISION = \'ecfc63c083900ce4f19daad695e711f38ab98fcf\'\nARMS = (\'base\', \'adapter\')\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef canonical(value):\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), ensure_ascii=False, allow_nan=False).encode()\n\n\ndef publish(path, value):\n    """Publish complete fsynced JSON without overwriting a prior receipt."""\n    path = Path(path)\n    raw = canonical(value) + b\'\\n\'\n    if path.exists():\n        if path.read_bytes() != raw:\n            raise ValueError(\'Preserve prior receipt: \' + path.name)\n        return\n    handle, temporary = tempfile.mkstemp(prefix=\'.capture-\', dir=path.parent)\n    try:\n        with os.fdopen(handle, \'wb\') as stream:\n            stream.write(raw)\n            stream.flush()\n            os.fsync(stream.fileno())\n        os.link(temporary, path)\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n\n\ndef verify_inputs(prompts, adapter, expected_sha):\n    prompts, adapter = Path(prompts), Path(adapter)\n    if prompts.is_symlink() or not prompts.is_file() or prompts.stat().st_size > 2 * 1024**2 or sha(prompts) != expected_sha:\n        raise ValueError(\'Frozen prompts differ\')\n    package = json.loads(prompts.read_bytes())\n    if (package.get(\'version\') != \'coding-repair-comparison020-v1\'\n            or package.get(\'base_model\') != BASE or package.get(\'base_revision\') != REVISION\n            or package.get(\'reference_material_included\') is not False):\n        raise ValueError(\'Unexpected comparison identity\')\n    expected_generation = {\'do_sample\': False, \'max_new_tokens\': 1024, \'context_tokens\': 4096,\n                           \'seed\': 42, \'max_time_seconds\': 180, \'calls_per_arm_case\': 1}\n    if package.get(\'generation\') != expected_generation:\n        raise ValueError(\'Frozen generation settings differ\')\n    ids = [case[\'instance_id\'] for case in package[\'cases\']]\n    if len(ids) != 12 or len(set(ids)) != 12 or any(not re.fullmatch(r\'[A-Za-z0-9_-]{1,160}\', value) for value in ids):\n        raise ValueError(\'Case membership differs\')\n    for name in [\'adapter_model.safetensors\', \'adapter_config.json\']:\n        path = adapter / name\n        expected = package[\'adapter_files\'][name]\n        if path.is_symlink() or not path.is_file() or path.stat().st_size != expected[\'bytes\'] or sha(path) != expected[\'sha256\']:\n            raise ValueError(\'Trained adapter differs: \' + name)\n    if json.loads((adapter / \'adapter_config.json\').read_bytes()).get(\'base_model_name_or_path\') != BASE:\n        raise ValueError(\'Adapter base differs\')\n    for case in package[\'cases\']:\n        if set(case[\'messages\'][0]) != {\'role\', \'content\'} or [m[\'role\'] for m in case[\'messages\']] != [\'system\', \'user\']:\n            raise ValueError(\'Only frozen system/user messages are allowed\')\n        if any(not isinstance(m[\'content\'], str) for m in case[\'messages\']):\n            raise ValueError(\'Invalid message content\')\n    return package\n\n\ndef tokenize_cases(package, tokenize):\n    prepared = []\n    for case in package[\'cases\']:\n        tokens = tokenize(case[\'messages\'])\n        unavailable = case.get(\'unavailable_reason\') if not case.get(\'context_available\') else None\n        if len(tokens) + package[\'generation\'][\'max_new_tokens\'] > package[\'generation\'][\'context_tokens\']:\n            unavailable = \'Whole prompt exceeds frozen context budget; no truncation or model call\'\n        prepared.append({**case, \'input_token_ids\': tokens, \'input_tokens\': len(tokens), \'unavailable_reason\': unavailable})\n    return prepared\n\n\ndef paired_run(package, cases, output, binding, generate):\n    """A crash after intent is unknown; captured outputs are reused, never regenerated."""\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    publish(output / \'binding.json\', binding)\n    records = []\n    for index, case in enumerate(cases):\n        # Counterbalance order without changing case membership or selecting on outcomes.\n        for arm in (ARMS if index % 2 == 0 else tuple(reversed(ARMS))):\n            stem = case[\'instance_id\'] + \'.\' + arm\n            result_path, intent_path = output / (stem + \'.json\'), output / (stem + \'.intent.json\')\n            if result_path.exists():\n                result = json.loads(result_path.read_bytes())\n                if (result.get(\'binding_sha256\') != hashlib.sha256(canonical(binding)).hexdigest()\n                        or result.get(\'capture_sha256\') != hashlib.sha256(canonical({k: v for k, v in result.items() if k != \'capture_sha256\'})).hexdigest()\n                        or (result.get(\'instance_id\'), result.get(\'arm\')) != (case[\'instance_id\'], arm)):\n                    raise ValueError(\'Captured result belongs to another configuration\')\n                records.append(result)\n                continue\n            if intent_path.exists():\n                raise RuntimeError(\'Unknown prior dispatch; never resend: \' + stem)\n            result = {\'instance_id\': case[\'instance_id\'], \'arm\': arm,\n                      \'binding_sha256\': hashlib.sha256(canonical(binding)).hexdigest(),\n                      \'input_tokens\': case[\'input_tokens\'], \'verified_repair\': None,\n                      \'verification_complete\': False}\n            if case.get(\'unavailable_reason\'):\n                result.update(status=\'unavailable\', reason=case[\'unavailable_reason\'], model_calls=0)\n            else:\n                publish(intent_path, {\'instance_id\': case[\'instance_id\'], \'arm\': arm,\n                                      \'binding_sha256\': result[\'binding_sha256\'],\n                                      \'prompt_sha256\': hashlib.sha256(canonical(case[\'messages\'])).hexdigest()})\n                started = time.monotonic()\n                try:\n                    generated = generate(case, arm)\n                    if not isinstance(generated[\'text\'], str) or len(generated[\'text\'].encode()) > 128 * 1024:\n                        raise ValueError(\'Oversized or invalid model output\')\n                    result.update(status=\'captured\', model_calls=1, **generated)\n                except Exception as error:\n                    # A dispatched inference error is preserved once; no alternate model/retry.\n                    result.update(status=\'inference_error\', model_calls=1, error_type=type(error).__name__,\n                                  reason=\'Inference raised; inspect captured runtime. No retry was sent.\')\n                result[\'elapsed_seconds\'] = time.monotonic() - started\n            result[\'capture_sha256\'] = hashlib.sha256(canonical(result)).hexdigest()\n            publish(result_path, result)\n            records.append(result)\n            print(json.dumps({\'instance_id\': case[\'instance_id\'], \'arm\': arm, \'status\': result[\'status\'],\n                              \'model_calls\': result[\'model_calls\']}), flush=True)\n    result = {\'version\': \'coding-generation-results020-v1\', \'binding\': binding, \'records\': records,\n              \'inference_complete\': len(records) == 2 * len(package[\'cases\']), \'repair_verification_complete\': False,\n              \'scope\': package[\'scope\'], \'training_runs\': 0}\n    publish(output / \'generations.json\', result)\n    return result\n\n\ndef run_gpu(prompts, adapter, output, expected_sha, scratch):\n    package = verify_inputs(prompts, adapter, expected_sha)\n    # A partially dispatched prior run must be diagnosed before allocating GPU or fetching weights.\n    output = Path(output)\n    if output.exists():\n        for intent in output.glob(\'*.intent.json\'):\n            if not intent.with_name(intent.name.replace(\'.intent.json\', \'.json\')).exists():\n                raise RuntimeError(\'Unknown prior dispatch; no model was loaded: \' + intent.name)\n        completed = output / \'generations.json\'\n        if completed.exists():\n            saved = json.loads(completed.read_bytes())\n            if saved.get(\'binding\', {}).get(\'prompts_sha256\') != expected_sha or saved.get(\'binding\', {}).get(\'runner_sha256\') != sha(__file__):\n                raise ValueError(\'Completed study configuration differs\')\n            for case in package[\'cases\']:\n                for arm in ARMS:\n                    if not (output / (case[\'instance_id\'] + \'.\' + arm + \'.json\')).is_file():\n                        raise ValueError(\'Completed study has a missing capture; never regenerate it\')\n            cases = tokenize_cases(package, lambda messages: [])\n            return paired_run(package, cases, output, saved[\'binding\'], lambda *_: (_ for _ in ()).throw(RuntimeError(\'Completed output must never dispatch\')))\n    if os.environ.get(\'KAIROS_EVALUATION_GPU\') != \'1\':\n        raise RuntimeError(\'Set KAIROS_EVALUATION_GPU=1 only in the approved free GPU evaluation runtime\')\n    import shutil\n    if shutil.disk_usage(scratch).free < 16 * 1024**3:\n        raise RuntimeError(\'Need 16 GiB cloud scratch free before the pinned model download\')\n    os.environ.update(HF_HUB_DISABLE_IMPLICIT_TOKEN=\'1\', WANDB_DISABLED=\'true\',\n                      TOKENIZERS_PARALLELISM=\'false\', PYTORCH_CUDA_ALLOC_CONF=\'expandable_segments:True\')\n    import torch\n    from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed\n    from peft import PeftModel\n    from huggingface_hub import snapshot_download\n    if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 14 * 1024**3:\n        raise RuntimeError(\'A CUDA GPU with at least 14 GiB is required; no CPU/Mac fallback\')\n    runtime = {name: importlib.metadata.version(name) for name in [\'torch\', \'transformers\', \'peft\', \'accelerate\', \'bitsandbytes\', \'tokenizers\']}\n    if any(runtime[name] != value for name, value in {\'transformers\': \'4.57.6\', \'peft\': \'0.21.2\', \'accelerate\': \'1.15.0\', \'bitsandbytes\': \'0.50.2\'}.items()):\n        raise RuntimeError(\'Reviewed inference dependency profile differs; stop before weights\')\n    model_path = Path(snapshot_download(BASE, revision=REVISION, cache_dir=str(Path(scratch) / \'public-model-cache\'),\n        token=False, allow_patterns=[\'*.json\', \'*.safetensors\', \'tokenizer.model\', \'*.txt\'], ignore_patterns=[\'*.bin\', \'*.pt\', \'*.pth\']))\n    model_files = {p.name: {\'bytes\': p.stat().st_size, \'sha256\': sha(p)} for p in model_path.iterdir() if p.is_file()}\n    tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True, trust_remote_code=False)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token_id = tokenizer.eos_token_id\n    for case in package[\'cases\']:\n        rendered = \'\'.join(\'<|im_start|>\' + message[\'role\'] + \'\\n\' + message[\'content\'] + \'<|im_end|>\\n\'\n                           for message in case[\'messages\']) + \'<|im_start|>assistant\\n\'\n        if rendered != tokenizer.apply_chat_template(case[\'messages\'], tokenize=False, add_generation_prompt=True):\n            raise ValueError(\'Pinned system/user template rendering differs; no inference dispatched\')\n    cases = tokenize_cases(package, lambda messages: tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True))\n    base = AutoModelForCausalLM.from_pretrained(str(model_path), local_files_only=True, trust_remote_code=False,\n        use_safetensors=True, device_map={\'\': 0}, torch_dtype=torch.float16, attn_implementation=\'sdpa\')\n    model = PeftModel.from_pretrained(base, str(adapter), is_trainable=False, local_files_only=True, use_safetensors=True)\n    model.eval()\n    if any(parameter.requires_grad for parameter in model.parameters()):\n        raise RuntimeError(\'Inference model unexpectedly has trainable parameters\')\n    if any(parameter.device.type != \'cuda\' for parameter in model.parameters()):\n        raise RuntimeError(\'CPU/disk offload is outside this profile\')\n    binding = {\'version\': \'coding-inference-binding020-v1\', \'prompts_sha256\': expected_sha,\n               \'runner_sha256\': sha(__file__), \'adapter_sha256\': package[\'adapter_files\'][\'adapter_model.safetensors\'][\'sha256\'],\n               \'runtime\': runtime, \'gpu\': torch.cuda.get_device_name(0), \'model_revision\': REVISION,\n               \'model_files\': model_files, \'generation\': package[\'generation\'], \'training_runs\': 0}\n    def generate(case, arm):\n        set_seed(package[\'generation\'][\'seed\'])\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n        ids = torch.tensor([case[\'input_token_ids\']], device=\'cuda\')\n        started = time.monotonic()\n        # Disabling adapters exposes the same frozen quantized base, with identical input and decoding.\n        with (model.disable_adapter() if arm == \'base\' else nullcontext()), torch.inference_mode():\n            values = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), do_sample=False,\n                max_new_tokens=package[\'generation\'][\'max_new_tokens\'], max_time=package[\'generation\'][\'max_time_seconds\'],\n                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, use_cache=True)\n        new = values[0, ids.shape[1]:].tolist()\n        eos = bool(new and new[-1] == tokenizer.eos_token_id)\n        return {\'text\': tokenizer.decode(new, skip_special_tokens=True), \'output_token_ids\': new,\n                \'output_tokens\': len(new), \'ended_with_eos\': eos,\n                \'stop_reason\': \'eos\' if eos else (\'output_budget\' if len(new) >= package[\'generation\'][\'max_new_tokens\'] else \'generation_time_limit\'),\n                \'generation_seconds\': time.monotonic() - started,\n                \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(),\n                \'peak_allocated_bytes\': torch.cuda.max_memory_allocated()}\n    return paired_run(package, cases, output, binding, generate)\n\n\ndef export_results(output):\n    import zipfile\n    output = Path(output)\n    result = json.loads((output / \'generations.json\').read_bytes())\n    expected = {\'binding.json\', \'generations.json\'}\n    for record in result[\'records\']:\n        stem = record[\'instance_id\'] + \'.\' + record[\'arm\']\n        expected.add(stem + \'.json\')\n        if record[\'model_calls\']:\n            expected.add(stem + \'.intent.json\')\n    files = {p.name: p for p in output.iterdir() if p.is_file() and p.suffix == \'.json\'}\n    if set(files) != expected or sum(p.stat().st_size for p in files.values()) > 4 * 1024**2:\n        raise ValueError(\'Unexpected or oversized export membership\')\n    manifest = {\'version\': \'coding-inference-export020-v1\', \'contains_model_weights\': False,\n                \'training_runs\': 0, \'repair_verification_complete\': False,\n                \'files\': {name: {\'bytes\': p.stat().st_size, \'sha256\': sha(p)} for name, p in sorted(files.items())}}\n    target = output.parent / \'kairos-evaluation020.zip\'\n    if not target.exists():\n        with zipfile.ZipFile(target, \'x\', zipfile.ZIP_DEFLATED) as archive:\n            for name, path in sorted(files.items()):\n                archive.write(path, name)\n            archive.writestr(\'manifest.json\', canonical(manifest))\n    with zipfile.ZipFile(target) as archive:\n        if len(archive.namelist()) != len(set(archive.namelist())) or set(archive.namelist()) != expected | {\'manifest.json\'}:\n            raise ValueError(\'Existing export membership differs\')\n        if json.loads(archive.read(\'manifest.json\')) != manifest:\n            raise ValueError(\'Existing export manifest differs; never regenerate it\')\n        for name, item in manifest[\'files\'].items():\n            raw = archive.read(name)\n            if len(raw) != item[\'bytes\'] or hashlib.sha256(raw).hexdigest() != item[\'sha256\']:\n                raise ValueError(\'Export byte hash differs\')\n    return {\'path\': str(target), \'bytes\': target.stat().st_size, \'sha256\': sha(target)}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--prompts\', type=Path, required=True)\n    parser.add_argument(\'--adapter\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--prompts-sha256\', required=True)\n    parser.add_argument(\'--scratch\', type=Path, default=Path(\'/content\'))\n    args = parser.parse_args()\n    run_gpu(args.prompts, args.adapter, args.output, args.prompts_sha256, args.scratch)\n\n\nif __name__ == \'__main__\':\n    main()\n'
RUNNER = WORK / 'coding_inference020.py'
if RUNNER.exists() and RUNNER.read_text() != RUNNER_SOURCE:
    raise RuntimeError('Runner changed; preserve this runtime for inspection.')
RUNNER.write_text(RUNNER_SOURCE)
child_env = dict(os.environ, KAIROS_EVALUATION_GPU='1', HF_HUB_DISABLE_IMPLICIT_TOKEN='1',
    WANDB_DISABLED='true', TOKENIZERS_PARALLELISM='false', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
child_env = {key: value for key, value in child_env.items() if key not in {'HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'OPENAI_API_KEY', 'GEMINI_API_KEY', 'GOOGLE_API_KEY'}}
args = [sys.executable, str(RUNNER), '--prompts', str(WORK / 'prompts.json'), '--prompts-sha256',
    '0cc835a7ad5d1e1a1e31dcad5e35cc2d65e81497936bd305ece9a7451ce5d511', '--adapter', str(WORK), '--output', str(WORK / 'results'), '--scratch', '/content']
result = subprocess.run(args, env=child_env, timeout=4800)
print({'inference_process_returncode': result.returncode, 'training_runs': 0})
if result.returncode:
    raise RuntimeError('Inference did not finish; preserve runtime and intents, do not resend.')


## Export evidence once

This small archive includes JSON captures and hashes only, never base weights or adapter weights. Existing identical archives are reused. Verification must finish locally before runtime cleanup. Inference completion is not a verified-repair score.


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('kairos_inference020', RUNNER)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
export = module.export_results(WORK / 'results')
print('KAIROS_EXPORT ' + json.dumps(export, sort_keys=True))
raw = Path(export['path']).read_bytes()
encoded = base64.b64encode(raw).decode()
display(HTML('<a download="kairos-evaluation020.zip" href="data:application/zip;base64,' + encoded + '">Download captured evaluation evidence</a>'))
# The visible fallback is bounded public evaluation evidence, with no weights or account data.
print('KAIROS_EXPORT_BASE64_BEGIN\n' + encoded + '\nKAIROS_EXPORT_BASE64_END')
